# SAC Training: Standard SAC with ActorMLP

Этот ноутбук обучает **SAC-агента (ActorMLP)** управлять камерой для Next-Best-View.

**Ключевая идея:** ActorMLP — и behavioral, и learned политика (стандартный SAC, без mismatch).  
ODIN NBV Head запускается на каждом шаге только как **источник признака `nbv_hint`** в `obs_vec[15:18]`.  
Coverage Head даёт сигнал награды: бинарный классификатор «есть ли скрытые объекты».

**Датасет сцен НЕ нужен** — среда PyBullet генерирует сцены динамически.

**Kaggle Inputs (подключить перед запуском):**
- Веса ODIN: загрузить `model_final.pth` как Kaggle Dataset (например `nbv-odin-weights`)
- *(Опционально)* Предыдущие checkpoints для resume

**Порядок запуска:** ячейки 1 → 2 → 3 → 4

## 1. Установка зависимостей (venv + ODIN стек)

In [ ]:
import os
import subprocess
import sys
import urllib.request

CONFIG = {
    "ODIN_DIR": "my_odin",
    "ODIN_REPO_URL": "https://github.com/SergKurchev/my_odin.git",
    "ODIN_BRANCH": "feature/nbv_dataset_process",
    "RL_REPO_URL": "https://github.com/SergKurchev/article-nbv.git",
    "RL_DIR": "nbv_rl",
    "ODIN_WEIGHTS_URL": "https://huggingface.co/katefgroup/odin/resolve/main/scannet_resnet_47.8_73.3_32k_1.5k.pth",
    "ODIN_WEIGHTS_PATH": "my_odin/models/odin_scannet_context.pth",
    "M2F_WEIGHTS_URL": "https://huggingface.co/katefgroup/odin/resolve/main/m2f_coco.pkl",
    "M2F_WEIGHTS_PATH": "my_odin/models/model_final_5c90d4.pkl",
}

# Создаём venv
if not os.path.exists("venv"):
    subprocess.run(["apt-get", "update", "-y"], check=False)
    subprocess.run(["apt-get", "install", "-y", "python3.10", "python3.10-venv",
                    "python3.10-dev", "python3.10-distutils",
                    "libgl1", "libglib2.0-0"], check=False)
    subprocess.run(["python3.10", "-m", "venv", "venv", "--without-pip"], check=True)
    urllib.request.urlretrieve("https://bootstrap.pypa.io/get-pip.py", "get-pip.py")
    subprocess.run(["venv/bin/python", "get-pip.py"], check=True)
    os.remove("get-pip.py")
    print("venv created")

VENV_PYTHON = os.path.abspath("venv/bin/python")
VENV_PIP = os.path.abspath("venv/bin/pip")

def make_venv_env(extra=None):
    env = os.environ.copy()
    venv_dir = os.path.abspath("venv")
    env["VIRTUAL_ENV"] = venv_dir
    env["PATH"] = os.path.join(venv_dir, "bin") + ":" + env.get("PATH", "")
    env.pop("PYTHONPATH", None)
    if extra:
        env.update(extra)
    if "RL_DIR" in CONFIG:
        env["PYTHONPATH"] = os.path.abspath(CONFIG["RL_DIR"]) + ":" + os.path.abspath(CONFIG["ODIN_DIR"])
    return env

def run_cmd(cmd, cwd=None, env=None, check=True):
    if cmd[0] == "pip": cmd[0] = VENV_PIP
    elif cmd[0] == "python": cmd[0] = VENV_PYTHON
    print(f">>> {' '.join(str(c) for c in cmd)}")
    subprocess.run(cmd, cwd=cwd, env=env, check=check)

def clean_build(directory):
    import shutil, glob
    for d in ["build", "dist"]:
        path = os.path.join(directory, d)
        if os.path.exists(path): shutil.rmtree(path)
    for egg in glob.glob(os.path.join(directory, "*.egg-info")):
        shutil.rmtree(egg)

def install_all():
    venv_env = make_venv_env()
    cuda_env = make_venv_env({"FORCE_CUDA": "1", "TORCH_CUDA_ARCH_LIST": "6.0;7.0;7.5;8.0;8.6"})

    # 0. Клонируем ODIN
    if not os.path.exists(CONFIG["ODIN_DIR"]):
        subprocess.run(["git", "clone", "-q", "-b", CONFIG["ODIN_BRANCH"],
                        CONFIG["ODIN_REPO_URL"], CONFIG["ODIN_DIR"]], check=True)

    # 0b. Клонируем RL репозиторий (article-nbv)
    if not os.path.exists(CONFIG["RL_DIR"]):
        subprocess.run(["git", "clone", "-q", CONFIG["RL_REPO_URL"], CONFIG["RL_DIR"]], check=True)

    # 1. PyTorch 2.2.0 + CUDA 12.1
    print("\n1. Installing PyTorch...")
    run_cmd(["pip", "install", "-q", "torch==2.2.0", "torchvision==0.17.0",
             "--index-url", "https://download.pytorch.org/whl/cu121"], env=venv_env)
    run_cmd(["pip", "install", "-q", "torch-scatter",
             "-f", "https://data.pyg.org/whl/torch-2.2.0+cu121.html"], env=venv_env)

    # 2. NumPy + Pillow
    print("\n2. Installing NumPy + Pillow...")
    run_cmd(["pip", "install", "-q", "numpy<2", "--force-reinstall"], env=venv_env)
    run_cmd(["pip", "install", "-q", "Pillow>=10.2.0"], env=venv_env)

    # 3. Фильтрация requirements.txt
    print("\n3. Cleaning ODIN requirements...")
    req_path = os.path.join(CONFIG["ODIN_DIR"], "requirements.txt")
    with open(req_path, 'r') as f: lines = f.readlines()
    with open(req_path, 'w') as f:
        for line in lines:
            lc = line.strip().lower()
            if any(x in lc for x in ["waspinator", "detectron2", "pytorch3d"]): continue
            if "pyyaml==5.3.1" in lc: f.write("pyyaml>=5.4.1\n")
            else: f.write(line)

    # 4. Build tools
    print("\n4. Build tools...")
    run_cmd(["pip", "install", "-q", "cython", "setuptools", "wheel", "pycocotools"], env=venv_env)

    # 5. ODIN requirements
    print("\n5. ODIN requirements...")
    run_cmd(["pip", "install", "-q", "-r", req_path], env=venv_env)
    run_cmd(["pip", "install", "-q", "ninja", "fvcore", "iopath"], env=venv_env)

    # 6. Detectron2
    print("\n6. Detectron2...")
    run_cmd(["pip", "install", "-q", "--no-build-isolation",
             "git+https://github.com/facebookresearch/detectron2.git"], env=venv_env)

    # 7. PyTorch3D
    print("\n7. PyTorch3D...")
    run_cmd(["pip", "install", "-q", "--no-build-isolation",
             "git+https://github.com/facebookresearch/pytorch3d.git"], env=cuda_env)

    # 8. NumPy + OpenCV pin
    print("\n8. Pinning NumPy + OpenCV...")
    run_cmd(["pip", "uninstall", "-y", "-q", "numpy"], env=venv_env)
    run_cmd(["pip", "install", "-q", "numpy==1.26.4"], env=venv_env)
    run_cmd(["pip", "install", "-q", "opencv-python-headless==4.8.0.76"], env=venv_env)

    # 9. pointops2 CUDA kernel
    print("\n9. pointops2...")
    pointops_dir = os.path.abspath(os.path.join(CONFIG["ODIN_DIR"], "libs", "pointops2"))
    clean_build(pointops_dir)
    run_cmd(["python", "setup.py", "install"], cwd=pointops_dir, env=cuda_env)

    # 10. Deformable attention
    print("\n10. Deformable attention...")
    deform_dir = os.path.abspath(os.path.join(CONFIG["ODIN_DIR"], "odin", "modeling", "pixel_decoder", "ops"))
    clean_build(deform_dir)
    run_cmd(["python", "setup.py", "build", "install"], cwd=deform_dir, env=cuda_env)

    # 11. RL зависимости
    print("\n11. RL dependencies (pybullet, gymnasium)...")
    run_cmd(["pip", "install", "-q",
             "pybullet", "gymnasium",
             "imageio", "pandas", "matplotlib"], env=venv_env)

    print("\n=== Installation complete ===")

install_all()

## 2. Поиск весов ODIN (датасет сцен НЕ нужен)

In [ ]:
ODIN_CFG = "my_odin/configs/scannet_context/3d.yaml"

# Ищем предобученные веса NBVActiveODIN
# Загрузите model_final.pth как Kaggle Dataset (напр. 'nbv-odin-weights')
POSSIBLE_WEIGHTS = [
    "/kaggle/input/notebooks/sergeistwpk/strawpick-segpoinnet-my-odin-nbv-2-active/output_nbv_stage2_active/model_final.pth",
    "/kaggle/input/nbv-odin-weights/model_final.pth",  # Kaggle Dataset
    "./output_nbv_stage2/model_final.pth",              # Из предыдущего запуска
    "./output_nbv_stage2/last_checkpoint.pth",
    "./output_odin_sac/best.pth",                       # Предыдущий SAC запуск
]

ODIN_WEIGHTS = None
for p in POSSIBLE_WEIGHTS:
    if os.path.exists(p):
        ODIN_WEIGHTS = p
        print(f"✓ Found ODIN weights: {p}")
        break

if ODIN_WEIGHTS is None:
    print("⚠ NBVActiveODIN weights not found! Downloading base ODIN weights...")
    ODIN_WEIGHTS = CONFIG["ODIN_WEIGHTS_PATH"]
    os.makedirs(os.path.dirname(ODIN_WEIGHTS), exist_ok=True)
    if not os.path.exists(ODIN_WEIGHTS):
        os.system(f"wget --tries=3 -q '{CONFIG['ODIN_WEIGHTS_URL']}' -O '{ODIN_WEIGHTS}'")

print(f"\nODIN_WEIGHTS = {ODIN_WEIGHTS}")
print(f"ODIN_CFG     = {ODIN_CFG}")
print(f"\nДатасет сцен НЕ нужен — PyBullet генерирует сцены динамически.")

## 3. Синхронизация checkpoint (для продолжения обучения)

In [ ]:
import shutil

OUTPUT_DIR = "./output_odin_sac"

def sync_previous_output(target_output=OUTPUT_DIR):
    """Копирует старые checkpoints из /kaggle/input."""
    os.makedirs(target_output, exist_ok=True)
    for root, dirs, files in os.walk("/kaggle/input"):
        if "output_odin_sac" in root or "output_rl" in root:
            pth_files = [f for f in files if f.endswith(".pth")]
            if pth_files:
                print(f"Found previous checkpoint in: {root}")
                for f in pth_files:
                    src = os.path.join(root, f)
                    dst = os.path.join(target_output, f)
                    if not os.path.exists(dst):
                        shutil.copy2(src, dst)
                        print(f"  Copied: {f}")
                return
    print("No previous checkpoints found. Training from scratch.")

sync_previous_output()

# Проверяем наличие best/last для информации
for f in ["best.pth", "last.pth"]:
    path = os.path.join(OUTPUT_DIR, f)
    if os.path.exists(path):
        print(f"✓ Found: {path}")

## 4. Запуск обучения: Standard SAC

ActorMLP — и behavioral policy (сбор данных), и learned policy (SAC-обновления).  
ODIN NBV Head заморожен; его выход передаётся как `nbv_hint` в obs — ActorMLP его видит.

**`--freeze_backbone`** замораживает весь ODIN backbone.  
Обучаются: **ActorMLP + Twin Q-critics + log\_std + log\_alpha**.

In [ ]:
RL_DIR = os.path.abspath(CONFIG["RL_DIR"])
ODIN_DIR = os.path.abspath(CONFIG["ODIN_DIR"])

# ====================================================================
# Конфигурация обучения — редактируйте параметры здесь
# ====================================================================
TOTAL_STEPS      = 200000   # Общее число шагов SAC
LR_ACTOR         = "1e-4"   # Learning rate ActorMLP (actor)
LR_CRITIC        = "3e-4"   # Learning rate Q-networks (critic)
BUFFER_SIZE      = "50000"  # Replay Buffer (больше → меньше корреляций в батче)
BATCH_SIZE       = "256"    # Batch для SAC updates (256 лучше использует P100)
LEARNING_STARTS  = "500"    # Шагов до начала SAC updates
GRADIENT_STEPS   = "4"      # SAC-апдейтов за 1 шаг среды (UTD ratio, ~4x GPU утилизация)
SCENE_STAGE      = "2"      # 1=single obj, 2=multi obj, 3=multi+obstacles
FREEZE_BACKBONE  = True     # True (рекомендуется): backbone заморожен, обучается ActorMLP + critics
TRAIN_LAST_TRANSFORMER_BLOCK = False  # Дополнительно разморозить последний блок трансформера ODIN
# ====================================================================

train_cmd = [
    VENV_PYTHON,
    f"{RL_DIR}/train_odin_sac_rl.py",

    # --- Веса и конфиг ODIN ---
    "--odin_weights", ODIN_WEIGHTS,
    "--odin_cfg", ODIN_CFG,

    # --- Параметры SAC ---
    "--total_steps", str(TOTAL_STEPS),
    "--lr_actor", LR_ACTOR,
    "--lr_critic", LR_CRITIC,
    "--buffer_size", BUFFER_SIZE,
    "--batch_size", BATCH_SIZE,
    "--learning_starts", LEARNING_STARTS,
    "--gradient_steps", GRADIENT_STEPS,

    # --- Конфигурация сцены ---
    "--scene_stage", SCENE_STAGE,
    "--num_classes", "24",

    # --- Output ---
    "--output_dir", OUTPUT_DIR,
    "--save_freq", "5000",
    "--log_freq", "10",
]

if FREEZE_BACKBONE:
    train_cmd.append("--freeze_backbone")
if TRAIN_LAST_TRANSFORMER_BLOCK:
    train_cmd.append("--train_last_transformer_block")

venv_env = make_venv_env({
    "PYTHONPATH": f"{RL_DIR}:{ODIN_DIR}",
    "PYBULLET_HEADLESS": "1",
    "DISPLAY": "",
})

print("=" * 60)
print("  SAC Training: ActorMLP = behavioral + learned policy")
print(f"  ODIN weights: {ODIN_WEIGHTS}")
print(f"  Total steps:  {TOTAL_STEPS}")
print(f"  Freeze:       {FREEZE_BACKBONE}")
print(f"  Scene stage:  {SCENE_STAGE}")
print(f"  Buffer: {BUFFER_SIZE}  Batch: {BATCH_SIZE}  Starts: {LEARNING_STARTS}  UTD: {GRADIENT_STEPS}")
print("=" * 60)
print()

subprocess.run(train_cmd, env=venv_env, check=True)


## Результаты

Файлы сохраняются в `./output_odin_sac/`:

| Файл | Описание |
|------|----------|
| `best.pth` | Лучшая политика (по суммарной награде за эпизод) |
| `last.pth` | Последний checkpoint |
| `ckpt_step*.pth` | Промежуточные checkpoints (каждые 5000 шагов) |
| `sac_metrics.csv` | Метрики: reward, p_hidden, critic/actor loss, alpha |
| `logs/training_metrics.csv` | Пошаговые метрики среды |

### Что содержит checkpoint (.pth)
```python
state = torch.load('best.pth')
state['actor_mlp']  # Веса ActorMLP (= политика, и behavioral и learned)
state['nbv_head']   # Веса NBV Head + Coverage Head (заморожены, сохраняются для inference)
state['critic']     # Twin Q-networks
state['log_std']    # Learnable exploration noise [6]
state['log_alpha']  # Entropy coefficient α
```

### Архитектура
```
ODIN Backbone (frozen) → scene_emb [256]
                               |
           NBV Head (frozen) → nbv_hint → obs_vec[15:18]
                               |
    obs_vec_full [274] = [v_t || scene_emb]
                               |
      ActorMLP (Actor) → next_camera_pose   ← обучается SAC (behavioral + learned)
      Coverage Head    → p_hidden           ← сигнал награды, обучается BCE
      Twin Q-Critics   → Q(s, a)            ← обучается SAC
```

### Первые ~20 эпизодов
ActorMLP стартует из случайной инициализации — ожидаются коллизии и OOB.  
Это нормально: Q быстро обучается штрафовать плохие действия, политика стабилизируется.